# Drafting Local (Wild Rift) — Fine-Tuning QLoRA + Exportación GGUF (`Q4_K_M`)

**Requisitos en Google Colab:**
1. Ve a **Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU T4** (gratis).
2. Ten a mano los 2 archivos generados en tu PC:
   - `local_ai/data/wr_draft_train.jsonl` (2,300 ejemplos)
   - `local_ai/data/wr_draft_val.jsonl` (200 ejemplos)
3. Ejecuta las celdas en orden. Tiempo estimado total en T4: **12 a 18 minutos**.

In [ ]:
# === 1. CONFIGURACIÓN DEL MODELO BASE ===
# Recomendado para Infinix Hot 40 Pro (Helio G99, 8GB RAM + Wild Rift abierto):
# - Qwen2.5-0.5B-Instruct (~400 MB en Q4_K_M, ~15-20 tok/s, ~600 MB RAM)
# Alternativas si tu benchmark en PocketPal AI mostró buen rendimiento con 1B-1.5B:
# - "unsloth/Llama-3.2-1B-Instruct-bnb-4bit" (~750 MB en Q4_K_M)
# - "unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit" (~980 MB en Q4_K_M)

BASE_MODEL = "unsloth/Qwen2.5-0.5B-Instruct-bnb-4bit"
MAX_SEQ_LENGTH = 512
QUANTIZATION_METHOD = "q4_k_m"  # Óptimo balance velocidad/calidad en ARM64 CPU
OUTPUT_NAME = "drafting_wr_local_v1"

print("Modelo base seleccionado:", BASE_MODEL)

In [ ]:
# === 2. INSTALAR UNSLOTH (QLoRA 2x más rápido + Exportación GGUF nativa) ===
%%capture
!pip install -q --upgrade unsloth trl datasets accelerate bitsandbytes

In [ ]:
# === 3. SUBIR DATASETS (wr_draft_train.jsonl y wr_draft_val.jsonl) ===
import os
from google.colab import files

if not (os.path.exists("wr_draft_train.jsonl") and os.path.exists("wr_draft_val.jsonl")):
    print("Selecciona wr_draft_train.jsonl y wr_draft_val.jsonl desde tu PC...")
    uploaded = files.upload()

assert os.path.exists("wr_draft_train.jsonl"), "Falta wr_draft_train.jsonl"
assert os.path.exists("wr_draft_val.jsonl"), "Falta wr_draft_val.jsonl"
print("OK: Datasets cargados correctamente.")

In [ ]:
# === 4. CARGAR MODELO BASE + APLICAR ADAPTADOR LoRA ===
import torch
from unsloth import FastLanguageModel
from datasets import load_dataset

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

def format_chatml(batch):
    texts = [
        tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)
        for msgs in batch["messages"]
    ]
    return {"text": texts}

train_ds = load_dataset("json", data_files="wr_draft_train.jsonl", split="train").map(format_chatml, batched=True)
val_ds = load_dataset("json", data_files="wr_draft_val.jsonl", split="train").map(format_chatml, batched=True)

print(f"Train: {len(train_ds)} | Val: {len(val_ds)}")
print("Muestra formateada:\n", train_ds[0]["text"][:500])

In [ ]:
# === 5. ENTRENAMIENTO QLoRA (SFTTrainer + Enfoque 100% en Respuestas Tácticas) ===
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    args=SFTConfig(
        dataset_text_field="text",
        max_seq_length=MAX_SEQ_LENGTH,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,  # Batch efectivo = 16
        warmup_steps=20,
        num_train_epochs=2,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=20,
        eval_strategy="steps",
        eval_steps=50,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=42,
        output_dir="outputs",
        report_to="none",
    ),
)

# Entrenar únicamente sobre los tokens de salida del asistente (no desperdiciar pesos en el JSON de entrada)
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)

stats = trainer.train()
print("Entrenamiento finalizado. Loss final:", stats.training_loss)

In [ ]:
# === 6. VALIDACIÓN REAL EN 5 DRAFTS DE PRUEBA (REGEX + CALIDAD) ===
import json, re

FastLanguageModel.for_inference(model)
reg_plan_a = re.compile(r"PLAN\s*A\s*:\s*([^•\n\r]+)", re.IGNORECASE)

with open("wr_draft_val.jsonl", "r", encoding="utf-8") as f:
    samples = [json.loads(next(f)) for _ in range(5)]

passed = 0
for i, sample in enumerate(samples, start=1):
    prompt_msgs = sample["messages"][:2]
    expected_top1 = json.loads(prompt_msgs[1]["content"])["top3"][0]["name"]

    inputs = tokenizer.apply_chat_template(
        prompt_msgs, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to("cuda")

    out = model.generate(
        input_ids=inputs,
        max_new_tokens=180,
        temperature=0.2,
        top_p=0.9,
        do_sample=True,
    )
    gen_text = tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True).strip()
    m = reg_plan_a.search(gen_text)
    ok = bool(m and m.group(1).strip() == expected_top1 and "**" not in gen_text)
    passed += int(ok)
    print(f"\n--- Prueba {i} ({'PASS' if ok else 'CHECK'}) | Esperado PLAN A: {expected_top1} ---")
    print(gen_text)

print(f"\nResultado QA en inferencia: {passed}/5 exactos.")

In [ ]:
# === 7. EXPORTAR Y CUANTIZAR A GGUF (Q4_K_M) PARA ANDROID ===
import glob, hashlib, os

model.save_pretrained_gguf(
    OUTPUT_NAME,
    tokenizer,
    quantization_method=QUANTIZATION_METHOD,
)

gguf_files = glob.glob(f"{OUTPUT_NAME}/*.gguf") + glob.glob(f"{OUTPUT_NAME}*/*.gguf") + glob.glob("*.gguf")
gguf_files = sorted(set(gguf_files))
print("Archivos GGUF generados:", gguf_files)

for gf in gguf_files:
    size_mb = os.path.getsize(gf) / (1024 * 1024)
    sha = hashlib.sha256(open(gf, "rb").read()).hexdigest()
    print(f" -> {gf} | {size_mb:.1f} MB | SHA-256: {sha}")

# Opcional: copiar a Google Drive para no perderlo y descargarlo fácil en tu Infinix Hot 40 Pro
# from google.colab import drive
# drive.mount('/content/drive')
# !cp {gguf_files[0]} /content/drive/MyDrive/drafting_wr_local_v1_q4_k_m.gguf